# Chapter 12 · Random Variables and Distributions: companion notebook
*Data Science and Artificial Intelligence in Health, Volume 1*

> **This notebook accompanies the book chapter; it does not reproduce it.** Definitions, derivations, and explanations are in the book. Here you run the calculations on the case and check the numbers. All data are **synthetic**, and the values are for teaching purposes only.

**Part A** follows the chapter's case, the record of a city's emergency call center, through the director's nine questions. **Part B** gives the written solution of every chapter exercise (statements are in the book) and, for the exercises whose solution needs a calculation, its code and result. Run the cells in order.

## Part A. The case

**Data.** `data/calls.csv`: 12,472 rows, one per call over 90 days, and six columns: `call_id`, `day`, `hour`, `second`, `wait_sec`, `ambulance` (yes/no). Each cell carries the number of the book section it belongs to.

### 12.1 Introduction: the data

In [1]:
# 1. Read the record: one row is one call
import numpy as np
import pandas as pd
from math import sqrt, log, exp
from scipy import stats

df = pd.read_csv("data/calls.csv")

# 2. Count rows and columns
rows, columns = df.shape
print("The record has", rows, "rows (one per call) and", columns, "columns.")
df.head()                                      # The first five rows

The record has 12472 rows (one per call) and 6 columns.


,call_id,day,hour,second,wait_sec,ambulance
0,C00001,1,0,1228,40,no
1,C00002,1,0,3090,253,no
2,C00003,1,0,3235,107,yes
3,C00004,1,1,3523,120,no
4,C00005,1,2,1180,91,no


### 12.2 Calls in the evening hour

*How many calls arrive between 18:00 and 18:59, and how much does that number vary from day to day?*

In [2]:
# 1. Calls between 18:00 and 18:59 on each of the 90 days (a day without a call counts as 0)
counts = df[df["hour"] == 18].groupby("day").size().reindex(range(1, 91), fill_value=0)

# 2. How many days had each count
days_by_count = counts.value_counts().sort_index()
display(days_by_count.to_frame("days"))

# 3. Expected value, variance, and standard deviation (the book divides by 90 days)
probability = days_by_count / 90
mean = (probability.index * probability).sum()
variance = (((probability.index - mean) ** 2) * probability).sum()
print("Mean:", round(mean, 2), "; variance:", round(variance, 2), "; standard deviation:", round(sqrt(variance), 2))
print("Range:", counts.min(), "to", counts.max())

# 4. The hour 19:00-19:59 and the sum of the two means
counts19 = df[df["hour"] == 19].groupby("day").size().reindex(range(1, 91), fill_value=0)
print("Mean of 19:00-19:59:", round(counts19.mean(), 2), "; sum of the two means:", round(counts.mean() + counts19.mean(), 2), "; mean of the daily totals:", round((counts + counts19).mean(), 2))

# 5. Days within one standard deviation of the mean (5 to 10 calls) and days with more than 13 calls
print("Days with 5 to 10 calls:", ((counts >= 5) & (counts <= 10)).sum(), "; days with more than 13 calls:", (counts > 13).sum())

,days
2,2
3,5
4,6
5,12
6,11
7,11
8,11
9,11
10,6
11,3


Mean: 7.52 ; variance: 8.45 ; standard deviation: 2.91
Range: 2 to 15
Mean of 19:00-19:59: 7.16 ; sum of the two means: 14.68 ; mean of the daily totals: 14.68
Days with 5 to 10 calls: 62 ; days with more than 13 calls: 2


### 12.3 A model for counts in an hour: the Poisson distribution

*How likely is a rush of 14 or more calls in that hour?*

In [3]:
# 1. Poisson probability of exactly 7 calls, with the mean of the 90 days
mu = counts.mean()
p7 = stats.poisson.pmf(7, mu)
print("P(X = 7):", round(p7, 3), "; expected days:", round(90 * p7, 1), "; observed days:", (counts == 7).sum())

# 2. Cumulative distribution function and the rush of 14 or more calls
f13 = stats.poisson.cdf(13, mu)
print("F(13):", round(f13, 4), "; P(X >= 14):", round(1 - f13, 4))

# 3. Days with 14 or more calls: expected and observed
print("Days with 14 or more calls: expected", round(90 * (1 - f13), 2), "; observed", (counts >= 14).sum())

# 4. Is the variance close to the mean? Ratio and dispersion test (chi-square, 89 degrees of freedom)
ratio = counts.var(ddof=0) / counts.mean()
statistic = ((counts - counts.mean()) ** 2).sum() / counts.mean()
print("Variance / mean:", round(ratio, 2), "; statistic:", round(statistic, 1), "; statistic / 89:", round(statistic / 89, 2), "; p =", round(1 - stats.chi2.cdf(statistic, 89), 2))

# 5. A fire: chance of 30 or more calls in the hour, under the Poisson model
chance30 = stats.poisson.sf(29, mu)
print("P(X >= 30):", chance30, "; about one in", round(1 / chance30 / 1e9, 1), "billion")

P(X = 7): 0.146 ; expected days: 13.2 ; observed days: 11
F(13): 0.978 ; P(X >= 14): 0.022
Days with 14 or more calls: expected 1.98 ; observed 2
Variance / mean: 1.12 ; statistic: 101.1 ; statistic / 89: 1.14 ; p = 0.18
P(X >= 30): 5.238861299148829e-10 ; about one in 1.9 billion


### 12.4 Ambulance or not: the Bernoulli and binomial distributions

*Of the next 12 calls, how many will need an ambulance?*

In [4]:
# 1. Fraction of calls with an ambulance
ambulance = df["ambulance"] == "yes"
p = ambulance.mean()
print("Ambulance sent for", ambulance.sum(), "of", len(df), "calls =", round(p, 4))
print("Variance of one call:", round(p * (1 - p), 4))

# 2. Mean, variance, and standard deviation for 12 calls
n = 12
print("Mean:", round(n * p, 2), "; variance:", round(n * p * (1 - p), 2), "; standard deviation:", round(sqrt(n * p * (1 - p)), 2))

# 3. Four ambulances free: more than four ambulance calls among 12
p5 = 1 - stats.binom.cdf(4, n, p)
print("P(X >= 5):", round(p5, 4))

# 4. Check with blocks of 12 consecutive calls
blocks = ambulance.values[: (len(df) // n) * n].reshape(-1, n).sum(axis=1)
print("Blocks:", len(blocks), "; with 5 or more:", (blocks >= 5).sum(), "=", round(100 * (blocks >= 5).mean(), 1), "%; the model expects about", round(len(blocks) * p5), "(", round(100 * p5, 1), "%)")

Ambulance sent for 2227 of 12472 calls = 0.1786
Variance of one call: 0.1467
Mean: 2.14 ; variance: 1.76 ; standard deviation: 1.33
P(X >= 5): 0.0475
Blocks: 1039 ; with 5 or more: 41 = 3.9 %; the model expects about 49 ( 4.7 %)


### 12.5 Recordings drawn without replacement: the hypergeometric distribution

*Suppose a supervisor listened to 10 recordings drawn at random from the 142 calls of day 1. In how many of these 10 was it necessary to send an ambulance?*

In [5]:
# 1. Day 1: calls and ambulance calls
day1 = df[df["day"] == 1]
N = len(day1)
K = (day1["ambulance"] == "yes").sum()
m = 10
print("Day 1:", N, "calls,", K, "with an ambulance")

# 2. One audit in detail: 10 recordings drawn at random, without replacement
rng = np.random.default_rng(12)
audit = day1.iloc[rng.choice(N, size=m, replace=False)].sort_values("call_id")
display(audit[["call_id", "hour", "wait_sec", "ambulance"]])
print("Ambulance calls among the 10 recordings:", (audit["ambulance"] == "yes").sum())

# 3. Four more audits: each draw gives a different list of calls
for k in range(2, 6):
    audit = day1.iloc[rng.choice(N, size=m, replace=False)]
    print("Audit", k, "- ambulance sent:", audit["ambulance"].tolist(), "- ambulance calls:", (audit["ambulance"] == "yes").sum())

# 4. Expected number and hypergeometric probabilities for 10 recordings
print("Expected ambulance calls among the 10:", round(m * K / N, 2))
print("P(X = 0):", round(stats.hypergeom.pmf(0, N, K, m), 3), "; P(X <= 1):", round(stats.hypergeom.cdf(1, N, K, m), 3), "; P(X >= 3):", round(1 - stats.hypergeom.cdf(2, N, K, m), 3))

# 5. The binomial with the same fraction, for comparison
print("Binomial P(X = 0):", round(stats.binom.pmf(0, m, K / N), 3), "; P(X >= 3):", round(1 - stats.binom.cdf(2, m, K / N), 3))
print("Variance: hypergeometric", round(stats.hypergeom.var(N, K, m), 2), "; binomial", round(m * (K / N) * (1 - K / N), 2))

# 6. Ten thousand audits: share of audits with each count, against the formula
is_ambulance = (day1["ambulance"] == "yes").to_numpy()
audits = np.array([is_ambulance[rng.choice(N, size=m, replace=False)].sum() for _ in range(10000)])
for x in range(6):
    print("x =", x, "; share of audits:", round((audits == x).mean(), 3), "; formula:", round(stats.hypergeom.pmf(x, N, K, m), 3))

Day 1: 142 calls, 25 with an ambulance


,call_id,hour,wait_sec,ambulance
8,C00009,5,71,no
25,C00026,9,143,no
26,C00027,9,102,no
27,C00028,9,157,yes
33,C00034,10,78,no
49,C00050,12,71,no
81,C00082,14,189,no
82,C00083,15,147,no
128,C00129,20,119,no
131,C00132,21,38,yes


Ambulance calls among the 10 recordings: 2
Audit 2 - ambulance sent: ['no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no'] - ambulance calls: 0
Audit 3 - ambulance sent: ['no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no'] - ambulance calls: 0
Audit 4 - ambulance sent: ['no', 'yes', 'no', 'no', 'yes', 'no', 'no', 'no', 'yes', 'no'] - ambulance calls: 3
Audit 5 - ambulance sent: ['no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no'] - ambulance calls: 0
Expected ambulance calls among the 10: 1.76
P(X = 0): 0.134 ; P(X <= 1): 0.445 ; P(X >= 3): 0.247
Binomial P(X = 0): 0.144 ; P(X >= 3): 0.251
Variance: hypergeometric 1.36 ; binomial 1.45
x = 0 ; share of audits: 0.139 ; formula: 0.134
x = 1 ; share of audits: 0.3 ; formula: 0.311
x = 2 ; share of audits: 0.308 ; formula: 0.308
x = 3 ; share of audits: 0.178 ; formula: 0.172
x = 4 ; share of audits: 0.06 ; formula: 0.06
x = 5 ; share of audits: 0.014 ; formula: 0.013


### 12.6 Waiting for the next ambulance call: the geometric and negative binomial distributions

*How many calls will pass until the next ambulance call, and until the third?*

In [6]:
# 1. Gaps between ambulance calls: calls from one ambulance call to the next (the next one included)
positions = np.where(ambulance.values)[0]
gaps_calls = np.diff(positions)
print("Gaps:", len(gaps_calls), "; mean:", round(gaps_calls.mean(), 2), "; variance:", round(gaps_calls.var(), 1), "; model mean:", round(1 / p, 2), "; model variance:", round((1 - p) / p ** 2, 1))

# 2. Geometric probabilities against the record
print("P(X = 1):", round(stats.geom.pmf(1, p), 3), "; record:", round((gaps_calls == 1).mean(), 3))
print("P(X <= 5):", round(stats.geom.cdf(5, p), 3), "; record:", round((gaps_calls <= 5).mean(), 3), "; P(X > 10):", round(stats.geom.sf(10, p), 3), "; record:", round((gaps_calls > 10).mean(), 3))

# 3. Calls until the third ambulance call: groups of three gaps (negative binomial, r = 3)
groups = gaps_calls[: len(gaps_calls) // 3 * 3].reshape(-1, 3).sum(axis=1)
print("Groups:", len(groups), "; mean:", round(groups.mean(), 2), "; model mean:", round(3 / p, 2), "; variance:", round(groups.var(), 1), "; model variance:", round(3 * (1 - p) / p ** 2, 1))
print("P(X <= 10):", round(stats.nbinom.cdf(10 - 3, 3, p), 3), "; record:", round((groups <= 10).mean(), 3), "; P(X > 25):", round(stats.nbinom.sf(25 - 3, 3, p), 3), "; record:", round((groups > 25).mean(), 3))

Gaps: 2226 ; mean: 5.6 ; variance: 25.2 ; model mean: 5.6 ; model variance: 25.8
P(X = 1): 0.179 ; record: 0.162
P(X <= 5): 0.626 ; record: 0.633 ; P(X > 10): 0.14 ; record: 0.137
Groups: 742 ; mean: 16.8 ; model mean: 16.8 ; variance: 73.6 ; model variance: 77.3
P(X <= 10): 0.259 ; record: 0.259 ; P(X > 25): 0.151 ; record: 0.158


### 12.7 When calls arrive: the uniform, exponential, and gamma distributions

*At what second of the hour does a call arrive, how long until the next call, and how long until the third?*

In [7]:
# 1. Second of arrival of the calls of 18:00-18:59 (uniform model)
seconds = df[df["hour"] == 18]["second"]
print("Calls:", len(seconds), "; mean second:", round(seconds.mean(), 1), "; uniform mean: 1800; first 15 minutes:", round(100 * (seconds < 900).mean(), 1), "%; model: 25%; third quarter hour:", round(100 * ((seconds >= 1800) & (seconds < 2700)).mean(), 1), "%")

# 2. Gaps between consecutive calls inside the hour, day by day (exponential model)
gap_list = []
for day, rows_of_day in df[df["hour"] == 18].groupby("day"):
    gap_list = gap_list + list(np.diff(np.sort(rows_of_day["second"].values)))
gaps = np.array(gap_list)
mean_gap = gaps.mean()
print("Gaps:", len(gaps), "; mean:", round(mean_gap, 1), "s")
print("P(gap > 600):", round(exp(-600 / mean_gap), 3), "; record:", round((gaps > 600).mean(), 3), "; P(gap <= 120):", round(1 - exp(-120 / mean_gap), 3), "; record:", round((gaps <= 120).mean(), 3), "; P(gap > 1200):", round(exp(-1200 / mean_gap), 3), "; record:", round((gaps > 1200).mean(), 3))

# 3. Time to the third call: sums of three consecutive gaps (gamma, shape 3)
triple_list = []
for day, rows_of_day in df[df["hour"] == 18].groupby("day"):
    day_gaps = np.diff(np.sort(rows_of_day["second"].values))
    n_triples = len(day_gaps) // 3
    if n_triples > 0:
        triple_list = triple_list + list(day_gaps[: n_triples * 3].reshape(-1, 3).sum(axis=1))
triples = np.array(triple_list)
print("Triples:", len(triples), "; mean:", round(triples.mean(), 1), "; model mean:", round(3 * mean_gap, 1), "; P(sum <= 1200):", round(stats.gamma.cdf(1200, 3, scale=mean_gap), 3), "; record:", round((triples <= 1200).mean(), 3))

# 4. If the calls arrived without interruption between hours (rate of the 90 days)
print("Mean gap at the hourly rate:", round(3600 / counts.mean(), 1), "s; three gaps:", round(3 * 3600 / counts.mean(), 1), "s; standard deviation of three gaps, model:", round(sqrt(3) * mean_gap, 1), "; triples in the record:", round(triples.std(ddof=0), 1))

Calls: 677 ; mean second: 1775.4 ; uniform mean: 1800; first 15 minutes: 26.6 %; model: 25%; third quarter hour: 24.1 %
Gaps: 587 ; mean: 413.5 s
P(gap > 600): 0.234 ; record: 0.233 ; P(gap <= 120): 0.252 ; record: 0.24 ; P(gap > 1200): 0.055 ; record: 0.06
Triples: 162 ; mean: 1179.6 ; model mean: 1240.5 ; P(sum <= 1200): 0.555 ; record: 0.58
Mean gap at the hourly rate: 478.6 s; three gaps: 1435.7 s; standard deviation of three gaps, model: 716.2 ; triples in the record: 650.6


### 12.8 The wait: a measurement that is not a count

*How long does a caller wait for an operator?*

In [8]:
# 1. Summary of the waits
w = df["wait_sec"]
print("Mean:", round(w.mean(), 1), "; standard deviation:", round(w.std(ddof=0), 1), "; median:", w.median(), "; longest:", w.max())

# 2. Fractions of calls answered within 60, 120, and 240 seconds
print("Waited 60 s or less:", round(100 * (w <= 60).mean(), 1), "%; 120 s or less:", round(100 * (w <= 120).mean(), 1), "%; 240 s or less:", round(100 * (w <= 240).mean(), 1), "%")

# 3. Waits of more than 240 seconds
print("More than 240 s:", (w > 240).sum(), "of", len(w), "=", round(100 * (w > 240).mean(), 2), "%")

# 4. Waits of 240 seconds or more, and of more than 420 seconds (outside the density figure)
print("240 s or more:", (w >= 240).sum(), "; more than 420 s:", (w > 420).sum())

Mean: 107.3 ; standard deviation: 71.0 ; median: 90.0 ; longest: 1137
Waited 60 s or less: 26.0 %; 120 s or less: 68.8 %; 240 s or less: 95.1 %
More than 240 s: 611 of 12472 = 4.9 %
240 s or more: 621 ; more than 420 s: 70


### 12.9 The normal distribution

*Which model gives the chance of a wait of more than four minutes?*

In [9]:
# 1. Normal model on the raw seconds
wait_mean = w.mean()
wait_sd = w.std(ddof=0)
print("Normal model: P(X < 0) =", round(stats.norm.cdf(0, wait_mean, wait_sd), 3), "; P(X > 240) =", round(stats.norm.sf(240, wait_mean, wait_sd), 3))

# 2. The next 200 calls: exact binomial and normal approximation, without and with the continuity correction
n200 = 200
mean200 = n200 * p
sd200 = sqrt(n200 * p * (1 - p))
print("Mean:", round(mean200, 2), "; standard deviation:", round(sd200, 2))
print("P(X >= 45): exact", round(1 - stats.binom.cdf(44, n200, p), 4), "; normal", round(stats.norm.sf(45, mean200, sd200), 4), "; with correction", round(stats.norm.sf(44.5, mean200, sd200), 4))

# 3. Blocks of 200 consecutive calls in the record
blocks200 = ambulance.values[: (len(df) // n200) * n200].reshape(-1, n200).sum(axis=1)
print("Blocks of 200:", len(blocks200), "; with 45 or more:", (blocks200 >= 45).sum(), "; mean:", round(blocks200.mean(), 1), "; standard deviation:", round(blocks200.std(), 1))

Normal model: P(X < 0) = 0.065 ; P(X > 240) = 0.031
Mean: 35.71 ; standard deviation: 5.42
P(X >= 45): exact 0.0554 ; normal 0.0432 ; with correction 0.0523
Blocks of 200: 62 ; with 45 or more: 2 ; mean: 35.7 ; standard deviation: 5.2


### 12.10 Skewed durations: the lognormal distribution

*Within what wait does the model estimate that nine callers in ten, or 19 in 20, are answered?*

In [10]:
# 1. The logarithm of the wait
lw = np.log(w)
log_mean = lw.mean()
log_sd = lw.std(ddof=0)
print("Log of the wait: mean", round(log_mean, 3), "; standard deviation", round(log_sd, 3), "; skewness", round(stats.skew(lw), 3))

# 2. Lognormal model: median, mean, standard score of 240 s, and P(Y > 240)
z = (log(240) - log_mean) / log_sd
print("Lognormal model: median", round(exp(log_mean), 1), "; mean", round(exp(log_mean + log_sd ** 2 / 2), 1), "; z =", round(z, 2), "; P(Y > 240) =", round(stats.norm.sf(z), 3))

# 3. Fraction of logarithms within 1 and 2 standard deviations of the mean
print("Within 1 standard deviation:", round(100 * ((lw > log_mean - log_sd) & (lw < log_mean + log_sd)).mean(), 1), "%; within 2:", round(100 * ((lw > log_mean - 2 * log_sd) & (lw < log_mean + 2 * log_sd)).mean(), 1), "%")

# 4. Quantiles: the wait exceeded by 10% and by 5% of the callers
print("0.90-quantile:", round(exp(log_mean + stats.norm.ppf(0.90) * log_sd), 1), "s; record:", round(w.quantile(0.90), 1), "; 0.95-quantile:", round(exp(log_mean + stats.norm.ppf(0.95) * log_sd), 1), "s; record:", round(w.quantile(0.95), 1))

Log of the wait: mean 4.496 ; standard deviation 0.599 ; skewness 0.009
Lognormal model: median 89.6 ; mean 107.2 ; z = 1.64 ; P(Y > 240) = 0.05
Within 1 standard deviation: 68.4 %; within 2: 95.3 %
0.90-quantile: 193.1 s; record: 192.9 ; 0.95-quantile: 240.1 s; record: 239.0


### 12.11 Three distributions that the tests will need

*Which distributions will the later chapters use to compare days or shifts?*

In [11]:
# 1. Draw 10,000 samples of 10 logarithms, with replacement (seed 12)
rng = np.random.default_rng(12)
draws = lw.values[rng.integers(0, len(lw), size=(10000, 10))]

# 2. Mean and standard deviation of each sample, and the standardized mean with sigma and with s
sample_mean = draws.mean(axis=1)
sample_sd = draws.std(axis=1, ddof=1)
z_known = (sample_mean - log_mean) / (log_sd / sqrt(10))
t_stat = (sample_mean - log_mean) / (sample_sd / sqrt(10))

# 3. Share of samples beyond the limits
print("Known sigma, beyond 1.96:", round(100 * (abs(z_known) > 1.96).mean(), 1), "%; with s, beyond 1.96:", round(100 * (abs(t_stat) > 1.96).mean(), 1), "%; with s, beyond 2.262:", round(100 * (abs(t_stat) > stats.t.ppf(0.975, 9)).mean(), 1), "%")

# 4. Chi-square and F quantities
chi_square = 9 * sample_sd ** 2 / log_sd ** 2
second_sample = lw.values[rng.integers(0, len(lw), size=(10000, 10))]
f_ratio = sample_sd ** 2 / second_sample.std(axis=1, ddof=1) ** 2
print("Chi-square above 16.92:", round(100 * (chi_square > stats.chi2.ppf(0.95, 9)).mean(), 1), "%; F above 3.18:", round(100 * (f_ratio > stats.f.ppf(0.95, 9, 9)).mean(), 1), "%")

# 5. Limits of the three distributions
print("Student t, 9 df, 97.5%:", round(stats.t.ppf(0.975, 9), 3), "; chi-square, 9 df, 95%:", round(stats.chi2.ppf(0.95, 9), 2), "; F, 9 and 9 df, 95%:", round(stats.f.ppf(0.95, 9, 9), 2))

Known sigma, beyond 1.96: 4.8 %; with s, beyond 1.96: 8.4 %; with s, beyond 2.262: 5.1 %
Chi-square above 16.92: 4.6 %; F above 3.18: 4.8 %
Student t, 9 df, 97.5%: 2.262 ; chi-square, 9 df, 95%: 16.92 ; F, 9 and 9 df, 95%: 3.18


## Part B. Exercise solutions

### 12.13 Exercise 1. Calls from 12:00 to 12:59

The variable is the number of calls in that hour on a given day, a discrete count. Over the 90 days the mean is 8.47 calls and the variance is 8.18, with counts from 4 to 18. The variance is close to the mean, as a Poisson distribution requires, so the model is plausible for this hour.

In [12]:
# 1. Calls between 12:00 and 12:59 on each day
counts12 = df[df["hour"] == 12].groupby("day").size().reindex(range(1, 91), fill_value=0)

# 2. Mean and variance of the counts
print("Mean:", round(counts12.mean(), 2), "; variance:", round(counts12.var(ddof=0), 2), "; range:", counts12.min(), "to", counts12.max())

Mean: 8.47 ; variance: 8.18 ; range: 4 to 18


### 12.13 Exercise 2. A distribution from a table

The probabilities are between 0 and 1 and add to $0.4+0.3+0.2+0.1=1$, so the table is a probability distribution. The expected value is $0(0.4)+1(0.3)+2(0.2)+3(0.1)=1.0$. The variance is $(0-1)^2(0.4)+(1-1)^2(0.3)+(2-1)^2(0.2)+(3-1)^2(0.1)=0.4+0+0.2+0.4=1.0$, and the standard deviation is 1.0.

### 12.13 Exercise 3. The Poisson model for 18:00 to 18:59

With $\mu=677/90$: $P(X=5)=0.1086$, $P(X\leq5)=0.2390$, and $P(X\geq10)=1-P(X\leq9)=0.2261$. On 21 of the 90 days (23.3%) the hour had 10 or more calls, close to the 22.6% of the model.

In [13]:
# 1. Poisson probabilities with the mean of 18:00-18:59
print("P(X = 5):", round(stats.poisson.pmf(5, mu), 4), "; P(X <= 5):", round(stats.poisson.cdf(5, mu), 4), "; P(X >= 10):", round(1 - stats.poisson.cdf(9, mu), 4))
print("Days with 10 or more calls:", (counts >= 10).sum(), "of 90 =", round(100 * (counts >= 10).mean(), 1), "%")

P(X = 5): 0.1086 ; P(X <= 5): 0.239 ; P(X >= 10): 0.2261
Days with 10 or more calls: 21 of 90 = 23.3 %


### 12.13 Exercise 4. Calls from 03:00 to 03:59

The mean is 1.87 calls (variance 2.18). With a Poisson model, $P(X=0)=e^{-168/90}\approx0.1546$ and $P(X\geq5)=0.0414$. In the record, 18 days (20.0%) had no call and 5 days (5.6%) had five or more. The model follows the record roughly, with fewer empty hours than observed.

In [14]:
# 1. Calls between 03:00 and 03:59 on each day
counts3 = df[df["hour"] == 3].groupby("day").size().reindex(range(1, 91), fill_value=0)
mu3 = counts3.mean()
print("Mean:", round(mu3, 2), "; variance:", round(counts3.var(ddof=0), 2))

# 2. Poisson probabilities of no call and of 5 or more calls, and the fractions of days
print("P(X = 0):", round(stats.poisson.pmf(0, mu3), 4), "; P(X >= 5):", round(1 - stats.poisson.cdf(4, mu3), 4))
print("Days with no call:", (counts3 == 0).sum(), "=", round(100 * (counts3 == 0).mean(), 1), "%; days with 5 or more:", (counts3 >= 5).sum(), "=", round(100 * (counts3 >= 5).mean(), 1), "%")

Mean: 1.87 ; variance: 2.18
P(X = 0): 0.1546 ; P(X >= 5): 0.0414
Days with no call: 18 = 20.0 %; days with 5 or more: 5 = 5.6 %


### 12.13 Exercise 5. Ambulances among 12 calls

The probability that an ambulance is sent is $2{,}227/12{,}472=0.1786$. With $n=12$, $P(X=0)=(1-p)^{12}=0.0944$ and $P(X\geq3)=0.3651$. Among the 1,039 blocks of 12 consecutive calls, 389 (37.4%) had three or more ambulance calls; the model expects about 379 (36.5%).

In [15]:
# 1. Binomial probabilities for 12 calls with the fraction of ambulance calls
print("p =", round(p, 4), "; P(X = 0):", round(stats.binom.pmf(0, 12, p), 4), "; P(X >= 3):", round(1 - stats.binom.cdf(2, 12, p), 4))

# 2. Blocks of 12 consecutive calls with 3 or more ambulance calls
print("Blocks with 3 or more:", (blocks >= 3).sum(), "of", len(blocks), "=", round(100 * (blocks >= 3).mean(), 1), "%; the model expects about", round(len(blocks) * (1 - stats.binom.cdf(2, 12, p))))

p = 0.1786 ; P(X = 0): 0.0944 ; P(X >= 3): 0.3651
Blocks with 3 or more: 389 of 1039 = 37.4 %; the model expects about 379


### 12.13 Exercise 6. Drawing without replacement

The 10 recordings are drawn without replacement, so each draw changes the share of ambulance calls left among the remaining calls, and the draws are not independent; the binomial distribution assumes the same $p$ in every draw. With $N=142$, $K=25$, and $n=10$, the hypergeometric probability is $P(X=2)=\binom{25}{2}\binom{117}{8}/\binom{142}{10}=0.3080$, and the binomial with $p=25/142$ gives 0.2963. They differ by 0.012 because 10 is small beside 142.

In [16]:
# 1. Hypergeometric and binomial probabilities of exactly 2 ambulance calls
print("Hypergeometric P(X = 2):", round(stats.hypergeom.pmf(2, N, K, m), 4), "; binomial P(X = 2):", round(stats.binom.pmf(2, m, K / N), 4))

Hypergeometric P(X = 2): 0.308 ; binomial P(X = 2): 0.2963


### 12.13 Exercise 7. Calls until the first ambulance call

The probability that the first ambulance call is the third call counted is $(1-p)^{2}p=0.8214^{2}\times0.1786=0.1205$. The probability of a gap of five calls or fewer is $P(X\leq5)=1-0.8214^{5}=0.626$, against 0.633 in the 2,226 gaps of the record, and $P(X>10)=0.8214^{10}=0.140$, against 0.137.

In [17]:
# 1. Geometric probabilities with p = 0.1786
print("P(X = 3):", round((1 - p) ** 2 * p, 4), "; P(X <= 5):", round(stats.geom.cdf(5, p), 3), "; record:", round((gaps_calls <= 5).mean(), 3), "; P(X > 10):", round(stats.geom.sf(10, p), 3), "; record:", round((gaps_calls > 10).mean(), 3))

P(X = 3): 0.1205 ; P(X <= 5): 0.626 ; record: 0.633 ; P(X > 10): 0.14 ; record: 0.137


### 12.13 Exercise 8. Calls until the second ambulance call

With $r=2$ the mean is $r/p=2/0.1786=11.2$ calls and the variance is $r(1-p)/p^{2}=51.5$. In the record the sums of two consecutive gaps average 11.20, equal to the model.

In [18]:
# 1. Calls until the second ambulance call: model and record
pairs = gaps_calls[: len(gaps_calls) // 2 * 2].reshape(-1, 2).sum(axis=1)
print("Model mean:", round(2 / p, 2), "; model variance:", round(2 * (1 - p) / p ** 2, 1), "; record mean:", round(pairs.mean(), 2))

Model mean: 11.2 ; model variance: 51.5 ; record mean: 11.2


### 12.13 Exercise 9. The second of arrival

For the 677 calls of the hour, the mean second is 1,775 and 24.2% arrived in the last 15 minutes. A uniform distribution from 0 to 3,600 gives a mean of 1,800 and 25%. The record is close to the uniform model.

In [19]:
# 1. Second of arrival in 18:00-18:59: mean and last 15 minutes
print("Mean second:", round(seconds.mean(), 1), "; last 15 minutes:", round(100 * (seconds >= 2700).mean(), 1), "%; uniform: 1800 and 25%")

Mean second: 1775.4 ; last 15 minutes: 24.2 %; uniform: 1800 and 25%


### 12.13 Exercise 10. Time between calls

With $m=413.5$ seconds, $P(\text{gap}>300)=e^{-300/413.5}=0.484$. In the 587 gaps of the record, 45.5% exceed 300 seconds. The model is about three percentage points high at 300 seconds and close beyond 600 seconds.

In [20]:
# 1. Gaps of more than 5 minutes
print("P(gap > 300):", round(exp(-300 / mean_gap), 3), "; record:", round((gaps > 300).mean(), 3))

P(gap > 300): 0.484 ; record: 0.455


### 12.13 Exercise 11. Fractions of the waits

In the record, 25.95% of the callers waited 60 seconds or less, 68.81% waited 120 seconds or less, and 2.15% (268 of 12,472) waited more than 300 seconds. The lognormal model gives 25.1%, 68.7%, and 2.2%: it matches the middle and the tail closely and is about one percentage point low at 60 seconds.

In [21]:
# 1. Fractions of the waits and the lognormal values
print("Record: 60 s or less", round(100 * (w <= 60).mean(), 2), "%; 120 s or less", round(100 * (w <= 120).mean(), 2), "%; more than 300 s", round(100 * (w > 300).mean(), 2), "%")
print("Lognormal: 60 s or less", round(100 * stats.norm.cdf((log(60) - log_mean) / log_sd), 1), "%; 120 s or less", round(100 * stats.norm.cdf((log(120) - log_mean) / log_sd), 1), "%; more than 300 s", round(100 * stats.norm.sf((log(300) - log_mean) / log_sd), 1), "%")

Record: 60 s or less 25.95 %; 120 s or less 68.81 %; more than 300 s 2.15 %
Lognormal: 60 s or less 25.1 %; 120 s or less 68.7 %; more than 300 s 2.2 %


### 12.13 Exercise 12. The normal model for the wait

A normal distribution with mean 107.3 and standard deviation 71.0 gives $P(X<0)=0.0653$ and $P(X>240)=0.0307$. A wait cannot be negative, so 6.5% below zero is impossible and shows that the bell shape does not fit this variable. The 3.1% above 240 seconds is also below the observed 4.9%.

In [22]:
# 1. Normal model with the mean and standard deviation of the waits
print("P(X < 0) =", round(stats.norm.cdf(0, wait_mean, wait_sd), 4), "; P(X > 240) =", round(stats.norm.sf(240, wait_mean, wait_sd), 4))

P(X < 0) = 0.0653 ; P(X > 240) = 0.0307


### 12.13 Exercise 13. The lognormal model

For 180 seconds, $\ln180=5.193$ and $z=(5.193-4.496)/0.599=1.16$, so $P(Y>180)=0.122$; in the record 11.9% waited more than 180 seconds. The wait exceeded by 1% of the callers is $e^{4.496+2.326\times0.599}=361$ seconds; in the record, the 99th percentile is 362.6 seconds.

In [23]:
# 1. Lognormal model: more than 180 seconds, and the 99th percentile
z180 = (log(180) - log_mean) / log_sd
print("z =", round(z180, 2), "; P(Y > 180) =", round(stats.norm.sf(z180), 3), "; observed:", round((w > 180).mean(), 3))
print("99th percentile of the model:", round(exp(log_mean + stats.norm.ppf(0.99) * log_sd), 1), "; of the record:", round(w.quantile(0.99), 1))

z = 1.16 ; P(Y > 180) = 0.122 ; observed: 0.119
99th percentile of the model: 361.1 ; of the record: 362.6


### 12.13 Exercise 14. The normal approximation for 200 calls

With $n=200$ and $p=0.1786$, the mean is 35.71 and the standard deviation 5.42. The exact probability of 40 or more is $P(X\geq40)=0.2390$. The normal approximation gives 0.2143 without the continuity correction (standardizing 40) and 0.2422 with it (standardizing 39.5): the correction brings the approximation much closer to the exact value.

In [24]:
# 1. 40 or more ambulance calls among 200 calls
print("Exact:", round(1 - stats.binom.cdf(39, n200, p), 4), "; normal:", round(stats.norm.sf(40, mean200, sd200), 4), "; with correction:", round(stats.norm.sf(39.5, mean200, sd200), 4))

Exact: 0.239 ; normal: 0.2143 ; with correction: 0.2422


### 12.13 Exercise 15. Standardized means of samples of 10

Of the 10,000 samples, 8.4% have $|t|>1.96$ and 5.1% have $|t|>2.262$. With 9 degrees of freedom the Student's $t$ limit, $\pm2.262$, must be used: the limit $\pm1.96$ belongs to the standard normal distribution, which applies when the true $\sigma$ is known and the sampling distribution of the mean is normal (approximately so in this example), and with it the sample would reject too often (8.4% instead of 5%).

In [25]:
# 1. Standardized mean of 10,000 samples of 10 logarithms (seed 12)
rng = np.random.default_rng(12)
draws = lw.values[rng.integers(0, len(lw), size=(10000, 10))]
t_stat = (draws.mean(axis=1) - log_mean) / (draws.std(axis=1, ddof=1) / sqrt(10))
print("Beyond 1.96:", round(100 * (abs(t_stat) > 1.96).mean(), 1), "%; beyond 2.262:", round(100 * (abs(t_stat) > stats.t.ppf(0.975, 9)).mean(), 1), "%")

Beyond 1.96: 8.4 %; beyond 2.262: 5.1 %


---
*Synthetic data (seed 12, `generate_data.py`). Teaching values only; nothing here is an operational recommendation or an estimate for a real service.*